In [7]:
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd

data_dir_candidates = [
    Path.cwd(),
    Path.cwd() / "llms_approach",
    Path.cwd().parent / "llms_approach",
]

for candidate in data_dir_candidates:
    gt_path = candidate / "nutrition5k_dataset_groundtruth.csv"
    pred_path = candidate / "foodqwen_macro_estimates.csv"
    if gt_path.exists() and pred_path.exists():
        data_dir = candidate
        break
else:
    raise FileNotFoundError("Could not locate the required CSV files. Check the paths and rerun the cell.")

groundtruth = pd.read_csv(data_dir / "nutrition5k_dataset_groundtruth.csv")
predictions = pd.read_csv(data_dir / "foodqwen_macro_estimates.csv")

predictions_renamed = predictions.rename(
    columns={
        "calories_kcal": "calories",
        "fat_g": "fat",
        "carbs_g": "carbs",
        "protein_g": "protein",
    }
)

merged = pd.merge(
    groundtruth,
    predictions_renamed,
    on="dish_id",
    how="inner",
    suffixes=("_actual", "_pred"),
)

overlap_ratio = len(merged) / len(groundtruth) if len(groundtruth) else np.nan

groundtruth_only = sorted(set(groundtruth["dish_id"]) - set(predictions_renamed["dish_id"]))
predictions_only = sorted(set(predictions_renamed["dish_id"]) - set(groundtruth["dish_id"]))

macro_columns = ["calories", "fat", "carbs", "protein"]
records = []

def safe_r2(actual_values: pd.Series, pred_values: pd.Series) -> float:
    diff = pred_values - actual_values
    denom = np.sum((actual_values - actual_values.mean()) ** 2)
    if denom == 0:
        return np.nan
    return 1 - np.sum(diff ** 2) / denom

def safe_corr(actual_values: pd.Series, pred_values: pd.Series) -> float:
    if actual_values.nunique() <= 1 or pred_values.nunique() <= 1:
        return np.nan
    return np.corrcoef(actual_values, pred_values)[0, 1]

for col in macro_columns:
    actual = merged[f"{col}_actual"].astype(float)
    pred = merged[f"{col}_pred"].astype(float)
    diff = pred - actual

    mae = np.mean(np.abs(diff))
    mse = np.mean(diff ** 2)
    rmse = np.sqrt(mse)
    mbe = np.mean(diff)

    denom = np.where(actual != 0, np.abs(actual), np.nan)
    mape = np.nanmean(np.abs(diff) / denom) * 100

    records.append(
        {
            "metric": col,
            "MAE": mae,
            "RMSE": rmse,
        }
    )

metrics_df = pd.DataFrame.from_records(records).set_index("metric").sort_index()

summary = pd.DataFrame(
    {
        "groundtruth_rows": [len(groundtruth)],
        "prediction_rows": [len(predictions_renamed)],
        "overlap_rows": [len(merged)],
        "overlap_ratio": [overlap_ratio],
        "groundtruth_only_ids": [len(groundtruth_only)],
        "prediction_only_ids": [len(predictions_only)],
    }
)

display(summary)
display(metrics_df.round(3))



,groundtruth_rows,prediction_rows,overlap_rows,overlap_ratio,groundtruth_only_ids,prediction_only_ids
0,3484,3484,3484,1.0,0,0


,MAE,RMSE
metric,,
calories,562.174,3345.048
carbs,419.689,3359.075
fat,73.911,924.750
protein,88.361,741.344
